# PromptSentinel — Notebook 1
## Dataset Preparation & Source Analysis

**Author:** Leesha Mogha </br>
**Institution:** IMS Ghaziabad (University Course Campus)  </br>
**Project:** PromptSentinel (v4)  </br>
**Builds on:** [Prompt-Safety-Classifier](https://github.com/leeeshart/Prompt-Safety-Classifier)  </br>

---

### What this notebook does
Collects prompt safety data from four public sources, cleans and
standardises them into a single dataset, and documents the
improvement over the v1/v2 data used in the previous research.

### Why this matters
The core finding from v3 was that **data quality matters more than
model sophistication.** This notebook directly addresses that —
building a dataset that is 18x larger and nearly balanced compared
to what v1/v2 used.

---
## Section 1: Setup
Installing libraries and logging into Hugging Face.
Some datasets require a free Hugging Face account to access.

In [ ]:
!pip install datasets pandas numpy -q

from huggingface_hub import login
from datasets import load_dataset
import pandas as pd
import numpy as np

login()

print("Libraries loaded and authenticated!")

Libraries loaded and authenticated!


---
## Section 2: Loading the Datasets

We combine four public datasets, each contributing something different:

| Source | Size | Why included |
|---|---|---|
| TrustAIRLab | 6,387 | Same source as v1/v2 — keeps results comparable |
| ToxicChat (lmsys) | 5,082 | Real user conversations — more realistic than curated data |
| Qualifire benchmark | 5,000 | Near-balanced — equal harmful and safe prompts |
| WildJailbreak (AllenAI) | 261,559 | Largest available — direct harmful + adversarial attacks |

### Dataset 1 — TrustAIRLab
The same dataset used in v1 and v2. Included here so results
stay comparable to previous versions. Collected from Reddit,
Discord and other platforms between 2022–2023.

**Known limitation:** Only 666 unsafe prompts out of 6,387 total (10.4%).
This imbalance was the main weakness of v1/v2.

In [ ]:
jailbreak = load_dataset(
    'TrustAIRLab/in-the-wild-jailbreak-prompts',
    'jailbreak_2023_05_07',
    split='train'
)
regular = load_dataset(
    'TrustAIRLab/in-the-wild-jailbreak-prompts',
    'regular_2023_05_07',
    split='train'
)

df1_unsafe = jailbreak.to_pandas()[['prompt']]
df1_safe   = regular.to_pandas()[['prompt']]

df1_unsafe['label']  = 'unsafe'
df1_safe['label']    = 'safe'
df1_unsafe['source'] = 'trustairlab'
df1_safe['source']   = 'trustairlab'

df1 = pd.concat([df1_unsafe, df1_safe], ignore_index=True)

print(f"TrustAIRLab loaded: {len(df1)} prompts")
print(df1['label'].value_counts())

TrustAIRLab loaded: 6387 prompts
label
safe      5721
unsafe     666
Name: count, dtype: int64


### Dataset 2 — ToxicChat (lmsys)
Real conversations collected from actual users of the Vicuna chatbot.
Human-annotated for toxicity. More realistic than manually curated
jailbreak lists because it reflects how real users actually behave.

In [ ]:
toxicchat = load_dataset('lmsys/toxic-chat', 'toxicchat0124', split='train')
df2 = toxicchat.to_pandas()

df2 = df2[['user_input', 'toxicity']].copy()
df2.columns = ['prompt', 'label']
df2['label']  = df2['label'].map({1: 'unsafe', 0: 'safe'})
df2['source'] = 'toxicchat'
df2 = df2.dropna()

print(f"ToxicChat loaded: {len(df2)} prompts")
print(df2['label'].value_counts())

ToxicChat loaded: 5082 prompts
label
safe      4698
unsafe     384
Name: count, dtype: int64


### Dataset 3 — Qualifire Benchmark
A purpose-built prompt injection benchmark with near-equal
safe and unsafe examples. Directly helps reduce the imbalance
problem that hurt v1/v2.

In [ ]:
qualifire = load_dataset('qualifire/prompt-injections-benchmark', split='test')
df3 = qualifire.to_pandas()

df3 = df3[['text', 'label']].copy()
df3.rename(columns={'text': 'prompt'}, inplace=True)
df3['label']  = df3['label'].map({'jailbreak': 'unsafe', 'benign': 'safe'})
df3['source'] = 'qualifire'
df3 = df3.dropna()

print(f"Qualifire loaded: {len(df3)} prompts")
print(df3['label'].value_counts())

Qualifire loaded: 5000 prompts
label
safe      2997
unsafe    2003
Name: count, dtype: int64


### Dataset 4 — WildJailbreak (AllenAI)
The largest public jailbreak dataset available. Contains four types
of prompts which map directly to the attack types studied in this research:

| data_type | Meaning | Our label |
|---|---|---|
| vanilla_harmful | Direct harmful request | unsafe |
| adversarial_harmful | Jailbreak-wrapped harmful request | unsafe |
| vanilla_benign | Normal safe prompt | safe |
| adversarial_benign | Looks suspicious but is actually safe | safe |

The adversarial vs vanilla split will be used in Notebook 2
to study whether different attack types need different detection.

In [ ]:
wild  = load_dataset(
    "allenai/wildjailbreak",
    "train",
    delimiter="\t",
    keep_default_na=False
)
df_wild = wild['train'].to_pandas()

print(f"WildJailbreak raw: {len(df_wild)} rows")
print(f"Columns: {df_wild.columns.tolist()}")
print()
print("Attack type breakdown:")
print(df_wild['data_type'].value_counts())

WildJailbreak raw: 261559 rows
Columns: ['vanilla', 'adversarial', 'completion', 'data_type']

Attack type breakdown:
data_type
adversarial_harmful    82728
adversarial_benign     78731
vanilla_harmful        50050
vanilla_benign         50050
Name: count, dtype: int64


#### Cleaning and standardising WildJailbreak labels
Mapping the four data_type values to our safe/unsafe system.

In [ ]:
df4 = df_wild[['vanilla', 'data_type']].copy()
df4.columns = ['prompt', 'label']

label_map = {
    'vanilla_harmful':     'unsafe',
    'adversarial_harmful': 'unsafe',
    'vanilla_benign':      'safe',
    'adversarial_benign':  'safe',
}

df4['label']  = df4['label'].map(label_map)
df4['source'] = 'wildjailbreak'
df4 = df4.dropna()
df4 = df4[df4['prompt'].str.strip() != '']

print(f"WildJailbreak loaded: {len(df4)} prompts")
print(df4['label'].value_counts())

WildJailbreak loaded: 261559 prompts
label
unsafe    132778
safe      128781
Name: count, dtype: int64


---
## Section 3: Combining & Cleaning

Merging all four sources into one dataset.
Duplicates are removed — if the same prompt appears in
multiple datasets, it is only kept once.

In [ ]:
df_combined = pd.concat([df1, df2, df3, df4], ignore_index=True)

df_combined = df_combined.drop_duplicates(subset='prompt')
df_combined = df_combined.dropna(subset=['prompt'])
df_combined = df_combined[df_combined['prompt'].str.strip() != '']

total    = len(df_combined)
unsafe_n = (df_combined['label'] == 'unsafe').sum()
safe_n   = (df_combined['label'] == 'safe').sum()

print("=" * 40)
print(f"FINAL TOTAL: {total:,} prompts")
print()
print(f"Safe:   {safe_n:,}  ({safe_n/total*100:.1f}%)")
print(f"Unsafe: {unsafe_n:,}  ({unsafe_n/total*100:.1f}%)")
print()
print("By source:")
print(df_combined.groupby(['source', 'label']).size().unstack(fill_value=0))

FINAL TOTAL: 116,202 prompts

Safe:   63,142  (54.3%)
Unsafe: 53,060  (45.7%)

By source:
label           safe  unsafe
source                      
qualifire       2997    1983
toxicchat       4607     374
trustairlab     5489     653
wildjailbreak  50049   50050


---
## Section 4: Improvement Over v1/v2

Comparing the dataset built here against what was used
in the previous research.

In [ ]:
comparison = pd.DataFrame([
    {
        'Version':         'v1 / v2',
        'Total prompts':   6387,
        'Unsafe prompts':  666,
        'Safe prompts':    5721,
        'Unsafe %':        '10.4%',
        'Sources':         1
    },
    {
        'Version':         'v4 (this notebook)',
        'Total prompts':   total,
        'Unsafe prompts':  unsafe_n,
        'Safe prompts':    safe_n,
        'Unsafe %':        f'{unsafe_n/total*100:.1f}%',
        'Sources':         df_combined['source'].nunique()
    }
])

print(comparison.to_string(index=False))
print()
print(f"Unsafe examples increased by {unsafe_n // 666}x")
print(f"Total dataset increased by {total // 6387}x")
print(f"Balance improved from 10.4% to {unsafe_n/total*100:.1f}%")

           Version  Total prompts  Unsafe prompts  Safe prompts Unsafe %  Sources
           v1 / v2           6387             666          5721    10.4%        1
v4 (this notebook)         116202           53060         63142    45.7%        4

Unsafe examples increased by 79x
Total dataset increased by 18x
Balance improved from 10.4% to 45.7%


---
## Section 5: Save

Saving the combined dataset for use in Notebook 2.

In [ ]:
df_combined.to_csv('combined_dataset_final.csv', index=False)

print("Saved: combined_dataset_final.csv")
print(f"Columns: {df_combined.columns.tolist()}")
print()

# Download to your machine so you don't lose it if Colab resets
from google.colab import files
files.download('combined_dataset_final.csv')

Saved: combined_dataset_final.csv
Columns: ['prompt', 'label', 'source']



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## Summary

| What | Result |
|---|---|
| Total prompts collected | 116,202 |
| Unsafe prompts | 53,060 (45.7%) |
| Safe prompts | 63,142 (54.3%) |
| Sources combined | 4 |
| Duplicates removed | Yes — from 277,662 raw to 116,202 final |

The dataset is now nearly balanced without any artificial tricks.
This directly addresses the core weakness identified in v1/v2.

**Next:** Notebook 2 — Attack Type Analysis
Will use the `data_type` column from WildJailbreak to study
whether different attack types need different detection methods.